## Import lib

In [34]:

import pandas as pd 
import numpy as np 
import os
import sys
sys.path.append('../')
sys.path.append('../..')
import random

from sklearn.metrics import classification_report, confusion_matrix \
        , roc_auc_score, f1_score, accuracy_score, precision_score, recall_score

from preprocessing.load_data import DataLoader

In [35]:
from detection.isolation_forest import IsolationForestDetector
from detection.lof import LocalOutlierFactorDetector
from detection.autoencoder import AutoencoderDetector

## Read data

In [36]:
training_data_path = '..\\..\\data\\raw\\SMD_data_use\\ServerMachineDataset\\train'

test_data_path = '..\\..\\data\\raw\\SMD_data_use\\ServerMachineDataset\\test'
test_labels_path = '..\\..\\data\\raw\\SMD_data_use\\ServerMachineDataset\\test_label'

Link tham khao 1 bai anomaly voi bo data set nay: https://www.kaggle.com/code/imenbenamar1/anomalydetection-multivariate-time-series   

Trong code có cả cell đặt tên cho cột  

In [37]:
new_column_names = [
    'cpu_r', 'load_1', 'load_5', 'load_15', 'mem_shmem', 'mem_u', 'mem_u_e', 'total_mem',
    'disk_q', 'disk_r', 'disk_rb', 'disk_svc', 'disk_u', 'disk_w', 'disk_wa', 'disk_wb',
    'si', 'so', 'eth1_fi', 'eth1_fo', 'eth1_pi', 'eth1_po', 'tcp_tw', 'tcp_use',
    'active_opens', 'curr_estab', 'in_errs', 'in_segs', 'listen_overflows', 'out_rsts',
    'out_segs', 'passive_opens', 'retransegs', 'tcp_timeouts', 'udp_in_dg', 'udp_out_dg',
    'udp_rcv_buf_errs', 'udp_snd_buf_errs'
]

In [38]:
training_data = pd.read_csv(training_data_path + '\\machine-1-1.txt', header=None)
print("Training data shape: ", training_data.shape)

training_data.head(2)

Training data shape:  (28479, 38)


,0,1,2,3,4,5,6,7,8,9,...,28,29,30,31,32,33,34,35,36,37
0,0.032258,0.039195,0.027871,0.024390,0.0,0.915385,0.343691,0.0,0.020011,0.000122,...,0.0,0.004298,0.029993,0.022131,0.0,0.000045,0.034677,0.034747,0.0,0.0
1,0.043011,0.048729,0.033445,0.025552,0.0,0.915385,0.344633,0.0,0.019160,0.001722,...,0.0,0.004298,0.030041,0.028821,0.0,0.000045,0.035763,0.035833,0.0,0.0


In [39]:
training_data.columns = new_column_names

training_data

,cpu_r,load_1,load_5,load_15,mem_shmem,mem_u,mem_u_e,total_mem,disk_q,disk_r,...,listen_overflows,out_rsts,out_segs,passive_opens,retransegs,tcp_timeouts,udp_in_dg,udp_out_dg,udp_rcv_buf_errs,udp_snd_buf_errs
0,0.032258,0.039195,0.027871,0.024390,0.0,0.915385,0.343691,0.0,0.020011,0.000122,...,0.0,0.004298,0.029993,0.022131,0.000000,0.000045,0.034677,0.034747,0.0,0.0
1,0.043011,0.048729,0.033445,0.025552,0.0,0.915385,0.344633,0.0,0.019160,0.001722,...,0.0,0.004298,0.030041,0.028821,0.000000,0.000045,0.035763,0.035833,0.0,0.0
2,0.043011,0.034958,0.032330,0.025552,0.0,0.915385,0.344633,0.0,0.020011,0.000122,...,0.0,0.004298,0.026248,0.021101,0.000000,0.000045,0.033012,0.033082,0.0,0.0
3,0.032258,0.028602,0.030100,0.024390,0.0,0.912821,0.342750,0.0,0.021289,0.000000,...,0.0,0.004298,0.030169,0.025733,0.000000,0.000022,0.035112,0.035182,0.0,0.0
4,0.032258,0.019068,0.026756,0.023229,0.0,0.912821,0.342750,0.0,0.018734,0.000000,...,0.0,0.004298,0.027240,0.022645,0.000000,0.000034,0.033447,0.033517,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28474,0.075269,0.046610,0.071349,0.076655,0.0,0.928205,0.269303,0.0,0.031649,0.000244,...,0.0,0.008596,0.068980,0.049408,0.000386,0.000034,0.064504,0.064572,0.0,0.0
28475,0.086022,0.070975,0.075808,0.077816,0.0,0.930769,0.269303,0.0,0.029946,0.000244,...,0.0,0.008596,0.073029,0.055584,0.000386,0.000034,0.067690,0.067757,0.0,0.0
28476,0.086022,0.065678,0.073579,0.076655,0.0,0.935897,0.270245,0.0,0.030372,0.000244,...,0.0,0.008596,0.070516,0.048893,0.000386,0.000034,0.064866,0.064934,0.0,0.0
28477,0.086022,0.056144,0.068004,0.074332,0.0,0.933333,0.271186,0.0,0.032643,0.000244,...,0.0,0.008596,0.070308,0.055069,0.000386,0.000045,0.067111,0.067178,0.0,0.0


In [40]:
training_data.describe()

,cpu_r,load_1,load_5,load_15,mem_shmem,mem_u,mem_u_e,total_mem,disk_q,disk_r,...,listen_overflows,out_rsts,out_segs,passive_opens,retransegs,tcp_timeouts,udp_in_dg,udp_out_dg,udp_rcv_buf_errs,udp_snd_buf_errs
count,28479.000000,28479.000000,28479.000000,28479.000000,28479.0,28479.000000,28479.000000,28479.0,28479.000000,28479.000000,...,28479.0,28479.000000,28479.000000,28479.000000,28479.000000,28479.000000,28479.000000,28479.000000,28479.0,28479.0
mean,0.064195,0.056882,0.053549,0.050188,0.0,0.913552,0.262274,0.0,0.020346,0.000159,...,0.0,0.005501,0.047294,0.036172,0.000188,0.000033,0.050351,0.050401,0.0,0.0
std,0.056685,0.042516,0.037695,0.037443,0.0,0.037972,0.066661,0.0,0.009982,0.001287,...,0.0,0.002545,0.033065,0.023716,0.000201,0.000015,0.035048,0.035044,0.0,0.0
min,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,...,0.0,0.000000,0.006146,0.006691,0.000000,0.000000,0.008760,0.008759,0.0,0.0
25%,0.032258,0.027542,0.027871,0.024390,0.0,0.907692,0.251412,0.0,0.012347,0.000000,...,0.0,0.004298,0.026128,0.021101,0.000000,0.000022,0.027945,0.028015,0.0,0.0
50%,0.043011,0.045551,0.042363,0.037166,0.0,0.912821,0.274011,0.0,0.021005,0.000000,...,0.0,0.004298,0.034858,0.027792,0.000000,0.000034,0.036415,0.036485,0.0,0.0
75%,0.086022,0.075212,0.070234,0.067364,0.0,0.920513,0.311676,0.0,0.027959,0.000122,...,0.0,0.007163,0.062250,0.044261,0.000386,0.000045,0.066025,0.066020,0.0,0.0
max,0.494624,0.421610,0.278707,0.250871,0.0,1.000000,0.360640,0.0,0.118081,0.140435,...,0.0,0.028653,0.275407,0.303139,0.002318,0.001271,0.311229,0.311206,0.0,0.0


In [63]:
from sklearn.preprocessing import StandardScaler

standard_scaler = StandardScaler()
training_data_numpy = standard_scaler.fit_transform(training_data) 

In [64]:
test_data = pd.read_csv(test_data_path + '\\machine-1-1.txt', header=None)
test_data.columns = new_column_names
print(test_data.shape)
test_data.head(2)

(28479, 38)


,cpu_r,load_1,load_5,load_15,mem_shmem,mem_u,mem_u_e,total_mem,disk_q,disk_r,...,listen_overflows,out_rsts,out_segs,passive_opens,retransegs,tcp_timeouts,udp_in_dg,udp_out_dg,udp_rcv_buf_errs,udp_snd_buf_errs
0,0.075269,0.065678,0.070234,0.074332,0.0,0.933333,0.274011,0.0,0.031081,0.000000,...,0.0,0.008596,0.068036,0.048893,0.000386,0.000034,0.064432,0.064500,0.0,0.0
1,0.086022,0.080508,0.075808,0.076655,0.0,0.930769,0.274953,0.0,0.031081,0.000122,...,0.0,0.008596,0.070020,0.050437,0.000386,0.000022,0.065228,0.065224,0.0,0.0


In [65]:
test_data_numpy = standard_scaler.transform(test_data)

d:\thac_si_phenika\master_thesis\Opstimus\venv_opstimus\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


In [66]:
test_label = pd.read_csv(test_labels_path + '\\machine-1-1.txt', header=None)

print(len(test_label))

28479


## LOF 

In [70]:
lof = LocalOutlierFactorDetector(n_neighbors=20, contamination=0.01, novelty=True)

lof.fit(training_data_numpy)

print("Model training completed.")

print("Scoring test data...")

lof_scores = lof.score(test_data_numpy)

lof_predictions = lof.predict(test_data_numpy)

print("Local Outlier Factor Scores: ", lof_scores[:5])
print("Local Outlier Factor Predictions: ", lof_predictions[:5])

Model training completed.
Scoring test data...
Local Outlier Factor Scores:  [-2.47319246 -2.53591411 -2.44896596 -2.55915943 -2.54554619]
Local Outlier Factor Predictions:  [ True  True  True  True  True]


In [71]:

print("Evaluating model performance...")

if test_label is not None:
    print("Number of anomalies detected: ", np.sum(lof_predictions))
    print("Percentage of anomalies detected: ", np.mean(lof_predictions) * 100, "%")
    print("Confusion Matrix:")
    print(confusion_matrix(test_label, lof_predictions.astype(int)))
    print("\nClassification Report:")
    print(classification_report(test_label, lof_predictions.astype(int)))

    ## vi score cua LOF la am. ví du score < -1 -> anomaly nen can dung -scores
    print("AUC-ROC of local outlier factor: ", roc_auc_score(test_label, -lof_scores))

    if not os.path.exists('../../models/checkpoint_SMD/'):
        os.makedirs('../../models/checkpoint_SMD/')

    result_folder = '../../models/checkpoint_SMD/baseline_lof/'
    if not os.path.exists(result_folder):
        os.makedirs(result_folder)

    lof.save_model(result_folder + 'lof.pkl')

    classification_report_df = classification_report(test_label, lof_predictions.astype(int), output_dict=True)

    report_df = pd.DataFrame(classification_report_df).transpose()
    report_df.to_csv(result_folder + 'lof_report.csv', index=True)

else:
    print("No labels available for evaluation.")
    print("Number of detected anomalies: ", np.sum(lof_predictions))

Evaluating model performance...
Number of anomalies detected:  28469
Percentage of anomalies detected:  99.96488640752835 %
Confusion Matrix:
[[    9 25776]
 [    1  2693]]

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.00      0.00     25785
           1       0.09      1.00      0.17      2694

    accuracy                           0.09     28479
   macro avg       0.50      0.50      0.09     28479
weighted avg       0.82      0.09      0.02     28479

AUC-ROC of local outlier factor:  0.8259793486743428


## Isolation forest

In [72]:
isolation_forest = IsolationForestDetector(contamination=0.01, random_state=42)

# vi isolation forest la model unsupervised, nen ta fit method tren test data
isolation_forest.fit(training_data_numpy)
# isolation_forest.fit(test_data_numpy) --- IGNORE ---
# isolation_forest.fit(test_data) --- IGNORE ---

isolation_forest_scores = isolation_forest.score(test_data_numpy)
isolation_forest_predictions = isolation_forest.predict(test_data_numpy)

print("Isolation Forest Scores: ", isolation_forest_scores[:5])
print("Isolation Forest Predictions: ", isolation_forest_predictions[:5])

Isolation Forest Scores:  [-0.18121883 -0.17626702 -0.18379477 -0.18149631 -0.18152367]
Isolation Forest Predictions:  [False False False False False]


In [74]:

print("Evaluating model performance...")

if test_label is not None:
    print("Number of anomalies detected: ", np.sum(isolation_forest_predictions))
    print("Percentage of anomalies detected: ", np.mean(isolation_forest_predictions) * 100, "%")
    print("Confusion Matrix:")
    print(confusion_matrix(test_label, isolation_forest_predictions.astype(int)))
    print("\nClassification Report:")
    print(classification_report(test_label, isolation_forest_predictions.astype(int)))

    print("AUC-ROC of isolation forest: ", roc_auc_score(test_label, isolation_forest_scores))

    if not os.path.exists('../../models/checkpoint_SMD/'):
        os.makedirs('../../models/checkpoint_SMD/')

    result_folder = '../../models/checkpoint_SMD/baseline_isolation_forest/'
    if not os.path.exists(result_folder):
        os.makedirs(result_folder)

    isolation_forest.save_model(result_folder + 'isolation_forest.pkl')

    classification_report_df = classification_report(test_label, isolation_forest_predictions.astype(int), output_dict=True)

    report_df = pd.DataFrame(classification_report_df).transpose()
    report_df.to_csv(result_folder + 'isolation_forest_report.csv', index=True)

else:
    print("No labels available for evaluation.")
    print("Number of detected anomalies: ", np.sum(isolation_forest_predictions))

Evaluating model performance...
Number of anomalies detected:  415
Percentage of anomalies detected:  1.4572140875732995 %
Confusion Matrix:
[[25702    83]
 [ 2362   332]]

Classification Report:
              precision    recall  f1-score   support

           0       0.92      1.00      0.95     25785
           1       0.80      0.12      0.21      2694

    accuracy                           0.91     28479
   macro avg       0.86      0.56      0.58     28479
weighted avg       0.90      0.91      0.88     28479

AUC-ROC of isolation forest:  0.8116337427925716


## Autoencoder

Can xem xem phan train co nhan hay k?   
Vi Autoencoder can tach duoc nhan 0 ra de training, sau do moi infer tren tap test


## RCA - Feature deviation

np.float64(0.17155745109402268)

In [51]:
mean_values = training_data.mean()
std_values = training_data.std()

mean_values

np.float64(0.17155745109402268)

In [ ]:
lof_predictions

test_label

lof_tp_data = test_data[(lof_predictions == 1) & (test_label[0] == 1)]
lof_tn_data = test_data[(lof_predictions == 0) & (test_label[0] == 0)]
lof_fp_data = test_data[(lof_predictions == 1) & (test_label[0] == 0)]
lof_fn_data = test_data[(lof_predictions == 0) & (test_label[0] == 1)]

print("True Positives: ", len(lof_tp_data))
print("True Negatives: ", len(lof_tn_data))
print("False Positives: ", len(lof_fp_data))
print("False Negatives: ", len(lof_fn_data))

True Positives:  2362
True Negatives:  19707
False Positives:  6078
False Negatives:  332


In [ ]:
# calculate feature deviation for true positives
lof_tp_deviation = (lof_tp_data - mean_values) / std_values

lof_tp_deviation.head(2)

print("Feature has >= 3 std from mean in true positives: ", 
      lof_tp_deviation[lof_tp_deviation.abs() >= 3].shape[0])

print("Feature has >= 5 std from mean in true positives: ", 
      lof_tp_deviation[lof_tp_deviation.abs() >= 5].shape[0])

Feature has >= 3 std from mean in true positives:  2362
Feature has >= 5 std from mean in true positives:  2362


In [ ]:
lof_tp_deviation.head(10)

,cpu_r,load_1,load_5,load_15,mem_shmem,mem_u,mem_u_e,total_mem,disk_q,disk_r,...,listen_overflows,out_rsts,out_segs,passive_opens,retransegs,tcp_timeouts,udp_in_dg,udp_out_dg,udp_rcv_buf_errs,udp_snd_buf_errs
15849,4.558299,0.755042,-0.178436,-0.347767,NaN,-0.491948,0.246697,NaN,1.217680,0.066159,...,NaN,3.467716,-0.217808,-0.201425,-0.936389,-0.717405,0.060953,0.059417,NaN,NaN
15850,7.593381,0.779950,-0.030565,-0.285727,NaN,-0.491948,0.260828,NaN,-0.758682,-0.123358,...,NaN,3.467716,-0.120062,-0.071216,0.982044,0.849628,0.093993,0.092461,NaN,NaN
15851,7.593381,0.381296,0.058146,-0.223686,NaN,-0.491948,0.260828,NaN,-0.531172,-0.123358,...,NaN,3.467716,-0.177161,-0.223140,-0.936389,-0.717405,0.023775,0.022235,NaN,NaN
15852,7.593381,0.057391,0.058146,-0.223686,NaN,-0.559471,0.246697,NaN,-0.758682,-0.123358,...,NaN,3.467716,-0.139417,-0.027827,-0.936389,0.849628,0.077473,0.078022,NaN,NaN
15853,7.593381,-0.191766,-0.030565,-0.223686,NaN,-0.559471,0.246697,NaN,-0.531172,-0.123358,...,NaN,3.467716,-0.256065,-0.353349,-0.936389,-0.717405,-0.003074,-0.002562,NaN,NaN
15854,7.593381,-0.316356,-0.119303,-0.254720,NaN,-0.559471,0.232581,NaN,-1.099897,-0.123358,...,NaN,3.467716,-0.187323,-0.136320,-0.936389,0.100177,0.048541,0.049087,NaN,NaN
15855,7.593381,-0.440922,-0.208015,-0.285727,NaN,-0.491948,0.232581,NaN,-0.858161,-0.037920,...,NaN,3.467716,-0.258969,-0.288244,-0.936389,0.849628,0.001034,0.001576,NaN,NaN
15856,7.593381,-0.440922,-0.267174,-0.285727,NaN,-0.491948,0.246697,NaN,-0.531172,-0.123358,...,NaN,3.467716,-0.226064,-0.114647,-0.936389,-0.717405,0.044432,0.042895,NaN,NaN
15857,7.213985,0.755042,0.146885,-0.130639,NaN,-0.559471,0.274959,NaN,3.094463,0.056839,...,NaN,3.467716,-0.310745,-0.201425,-0.936389,0.100177,0.013446,0.013960,NaN,NaN
15858,7.593381,0.057391,0.058146,-0.161646,NaN,-0.559471,0.274959,NaN,3.023334,-0.123358,...,NaN,3.467716,-0.190710,-0.201425,-0.936389,0.100177,0.019638,0.020181,NaN,NaN


In [ ]:
lof_tp_deviation[lof_tp_deviation.abs() >= 3]

,cpu_r,load_1,load_5,load_15,mem_shmem,mem_u,mem_u_e,total_mem,disk_q,disk_r,...,listen_overflows,out_rsts,out_segs,passive_opens,retransegs,tcp_timeouts,udp_in_dg,udp_out_dg,udp_rcv_buf_errs,udp_snd_buf_errs
15849,4.558299,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,3.467716,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15850,7.593381,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,3.467716,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15851,7.593381,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,3.467716,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15852,7.593381,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,3.467716,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15853,7.593381,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,3.467716,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24680,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,84.578214,NaN,...,NaN,5.719077,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24681,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,5.719077,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
26114,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,67.388534,NaN,...,NaN,10.222584,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27554,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,76.729869,NaN,...,NaN,10.222584,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [1]:
import numpy as np
import pandas as pd

def calculate_psi(expected, actual, bins=10):
    # Create bins based on expected data
    breakpoints = np.percentile(expected, np.arange(0, 100 + 100 / bins, 100 / bins))
    
    # Bucketize data
    expected_bins = np.histogram(expected, bins=breakpoints)[0]
    actual_bins = np.histogram(actual, bins=breakpoints)[0]
    
    # Convert to proportions
    expected_perc = expected_bins / len(expected)
    actual_perc = actual_bins / len(actual)
    
    # Avoid division by zero
    expected_perc = np.where(expected_perc == 0, 0.0001, expected_perc)
    actual_perc = np.where(actual_perc == 0, 0.0001, actual_perc)
    
    # Calculate PSI
    psi = np.sum((expected_perc - actual_perc) * np.log(expected_perc / actual_perc))
    
    return psi

In [2]:
train_data = np.random.normal(0, 1, 1000)
prod_data = np.random.normal(0.5, 1.2, 1000)

psi_value = calculate_psi(train_data, prod_data)

print("PSI:", psi_value)

PSI: 0.20478815296344913
